# Feature Engineering — DataCo SMART SUPPLY CHAIN

Transform the raw DataCo dataset into a feature-rich dataframe for predictive modelling of **cold-chain simulation**, **logistics risk**, and **quality degradation**.

| Step | Feature | Formula |
|------|---------|----------|
| 1 | Preprocessing | Drop IDs, PII, image links, target leakage |
| 2 | Delay | `Days real − Days scheduled` |
| 3 | Distance | Haversine to warehouse (39.8283, −98.5795) |
| 4 | Normalization | sklearn `MinMaxScaler` |
| 5 | RouteRisk | `0.4·Delay_norm + 0.3·Distance_norm + 0.3·Late_delivery_risk` |
| 6 | TempDev | `0.002·Distance_norm + 0.5·Delay_norm + 0.3·RouteRisk` |
| 7 | QualityDegradation | `Q0 · exp(-λ · (Delay_norm + TempDev))` |
| 8 | RefrigerationCost | `Distance × ShippingModeFactor` |

---
## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")

---
## 2. Configuration & Constants

In [ ]:
# --- Paths ---
DATA_PATH   = "./data/DataCoSupplyChainDataset.csv"
OUTPUT_PATH = "./data/cold_chain_supply_chain.csv"

# --- Reference warehouse (geographic centre of contiguous US) ---
WAREHOUSE_LAT = 39.8283
WAREHOUSE_LON = -98.5795

# --- Earth's mean radius (WGS-84) ---
EARTH_RADIUS_KM = 6_371.0

# --- Quality degradation model ---
Q0     = 100    # Initial product quality (0-100 scale)
LAMBDA = 0.05   # Decay constant

# --- Refrigeration cost multiplier per shipping mode ---
# Faster modes require more aggressive refrigeration -> higher cost factor
REFRIGERATION_FACTORS = {
    "Same Day"       : 3.0,
    "First Class"    : 2.5,
    "Second Class"   : 2.0,
    "Standard Class" : 1.0,
}

# --- Columns to drop (IDs, PII, image links, target leakage) ---
COLUMNS_TO_DROP = [
    "Customer Email", "Customer Fname", "Customer Lname",
    "Customer Password", "Customer Street", "Customer Zipcode",
    "Customer Id", "Order Customer Id", "Order Id",
    "Order Item Id", "Order Item Cardprod Id",
    "Product Card Id", "Product Image", "Product Description",
    "Product Status",       # sparse / non-informative
    "Category Id",          # redundant with Category Name
    "Department Id",        # redundant with Department Name
    "Product Category Id",  # redundant with Category Name
    "Order Zipcode",        # redundant with Order City/State
    "Delivery Status",      # target leakage (derived from Late_delivery_risk)
]

print(f"Config loaded. Will drop {len(COLUMNS_TO_DROP)} irrelevant columns.")

---
## 3. Load & Preprocess

In [ ]:
# Load raw dataset
df = pd.read_csv(DATA_PATH, encoding_errors="ignore")
print(f"Raw shape: {df.shape}")
print(f"Columns : {list(df.columns)}")

In [ ]:
# Drop irrelevant columns (only those that exist in the dataframe)
cols_present = [c for c in COLUMNS_TO_DROP if c in df.columns]
df.drop(columns=cols_present, inplace=True)

print(f"Dropped {len(cols_present)} columns")
print(f"Remaining shape: {df.shape}")
df.head(3)

---
## 4. Delay Calculation

$$\text{Delay} = \text{Days for shipping (real)} - \text{Days for shipment (scheduled)}$$

Positive delay means the shipment arrived **later** than scheduled (SLA breach).

In [ ]:
df["Delay"] = (
    df["Days for shipping (real)"] - df["Days for shipment (scheduled)"]
)

print(f"Delay range: [{df['Delay'].min()}, {df['Delay'].max()}]")
print(f"Mean delay : {df['Delay'].mean():.4f} days")
df["Delay"].value_counts().sort_index()

---
## 5. Distance Calculation (Haversine Formula)

Great-circle distance between each customer location and the reference warehouse at **(39.8283, -98.5795)**.

$$a = \sin^2\!\left(\frac{\Delta\varphi}{2}\right) + \cos(\varphi_1)\cos(\varphi_2)\sin^2\!\left(\frac{\Delta\lambda}{2}\right)$$
$$d = 2R \cdot \arcsin\!\left(\sqrt{a}\right)$$

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    """
    Compute great-circle distance using the Haversine formula.
    
    Parameters
    ----------
    lat1, lon1 : array-like  — Point(s) 1 in DEGREES
    lat2, lon2 : array-like  — Point(s) 2 in DEGREES
    
    Returns
    -------
    np.ndarray — Distance(s) in kilometres
    """
    # Degrees -> Radians
    phi1, lam1, phi2, lam2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    d_phi = phi2 - phi1
    d_lam = lam2 - lam1
    
    a = (
        np.sin(d_phi / 2.0) ** 2
        + np.cos(phi1) * np.cos(phi2) * np.sin(d_lam / 2.0) ** 2
    )
    c = 2.0 * np.arcsin(np.sqrt(a))
    
    return EARTH_RADIUS_KM * c

print("Haversine function defined.")

In [ ]:
df["Distance"] = haversine(
    df["Latitude"].values,
    df["Longitude"].values,
    WAREHOUSE_LAT,
    WAREHOUSE_LON,
)

print(f"Distance range: [{df['Distance'].min():.2f}, {df['Distance'].max():.2f}] km")
print(f"Mean distance : {df['Distance'].mean():.2f} km")
df["Distance"].describe()

---
## 6. Normalization (MinMaxScaler)

Scale **Delay** (clipped at 0, so early arrivals don't skew the range) and **Distance** to [0, 1].

In [ ]:
scaler = MinMaxScaler()

# Clip Delay >= 0 before scaling
delay_clipped = df["Delay"].clip(lower=0).values.reshape(-1, 1)
distance_vals = df["Distance"].values.reshape(-1, 1)

# Fit-transform both features together
scaled = scaler.fit_transform(
    np.hstack([delay_clipped, distance_vals])
)

df["Delay_norm"]    = scaled[:, 0]
df["Distance_norm"] = scaled[:, 1]

print("Normalization complete.")
df[["Delay", "Delay_norm", "Distance", "Distance_norm"]].describe().round(4)

---
## 7. Route Risk Index

Composite risk score combining delay severity, geographic exposure, and late-delivery flag:

$$\text{RouteRisk} = 0.4 \times \text{Delay}_{\text{norm}} + 0.3 \times \text{Distance}_{\text{norm}} + 0.3 \times \text{Late\_delivery\_risk}$$

In [ ]:
df["RouteRisk"] = (
    0.4 * df["Delay_norm"]
    + 0.3 * df["Distance_norm"]
    + 0.3 * df["Late_delivery_risk"]
)

print(f"RouteRisk range: [{df['RouteRisk'].min():.4f}, {df['RouteRisk'].max():.4f}]")
df["RouteRisk"].describe()

---
## 8. Cold Chain Simulation

### 8a. Temperature Deviation
Proxy for how far cargo temperature drifts from the ideal set-point:

$$\text{TempDev} = 0.002 \times \text{Distance}_{\text{norm}} + 0.5 \times \text{Delay}_{\text{norm}} + 0.3 \times \text{RouteRisk}$$

### 8b. Quality Degradation (Exponential Decay)
First-order kinetic degradation model (Labuza & Riboh, 1982):

$$Q(t) = Q_0 \cdot e^{-\lambda \cdot (\text{Delay}_{\text{norm}} + \text{TempDev})}$$

where $Q_0 = 100$ and $\lambda = 0.05$.

In [ ]:
# Temperature Deviation
df["TempDev"] = (
    0.002 * df["Distance_norm"]
    + 0.5  * df["Delay_norm"]
    + 0.3  * df["RouteRisk"]
)

print(f"TempDev range: [{df['TempDev'].min():.4f}, {df['TempDev'].max():.4f}]")
df["TempDev"].describe()

In [ ]:
# Quality Degradation (exponential decay)
df["QualityDegradation"] = (
    Q0 * np.exp(-LAMBDA * (df["Delay_norm"] + df["TempDev"]))
).clip(lower=0, upper=100)

print(f"Quality range: [{df['QualityDegradation'].min():.4f}, {df['QualityDegradation'].max():.4f}]")
df["QualityDegradation"].describe()

---
## 9. Refrigeration Cost

$$\text{RefrigerationCost} = \text{Distance} \times \text{Factor}_{\text{mode}}$$

| Shipping Mode | Factor | Rationale |
|---------------|--------|-----------|
| Same Day | 3.0 | Tightest cold-chain tolerance |
| First Class | 2.5 | High-priority active cooling |
| Second Class | 2.0 | Standard active cooling |
| Standard Class | 1.0 | Passive / minimal cooling |

In [ ]:
df["RefrigerationCost"] = (
    df["Distance"]
    * df["Shipping Mode"].map(REFRIGERATION_FACTORS)
)

print("Refrigeration cost by shipping mode:")
df.groupby("Shipping Mode")["RefrigerationCost"].describe().round(2)

---
## 10. Final Dataset — Export & Summary

In [ ]:
# Final column selection
FINAL_OUTPUT_COLUMNS = [
    # --- Original columns ---
    "order date (DateOrders)",
    "Order Item Quantity",
    "Sales",
    "Shipping Mode",
    "Market",
    "Category Name",
    "Order Region",
    "Product Price",
    "Latitude",
    "Longitude",
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Late_delivery_risk",
    # --- Engineered features ---
    "Delay",
    "Distance",
    "Delay_norm",
    "Distance_norm",
    "RouteRisk",
    "TempDev",
    "QualityDegradation",
    "RefrigerationCost",
]

# Verify all columns exist
missing = [c for c in FINAL_OUTPUT_COLUMNS if c not in df.columns]
if missing:
    raise KeyError(f"Missing columns: {missing}")

df_final = df[FINAL_OUTPUT_COLUMNS].copy()
print(f"Final shape: {df_final.shape}")
df_final.head()

In [ ]:
# Save to CSV
df_final.to_csv(OUTPUT_PATH, index=False)
print(f"Dataset saved to: {OUTPUT_PATH}")
print(f"Shape: {df_final.shape}")

In [ ]:
# Derived feature statistics
derived = [
    "Delay", "Distance", "Delay_norm", "Distance_norm",
    "RouteRisk", "TempDev", "QualityDegradation", "RefrigerationCost",
]

print("=== Derived Feature Statistics ===")
display(df_final[derived].describe().round(4))

In [ ]:
# Mean of derived features by Market
print("=== Mean by Market ===")
display(df_final.groupby("Market")[derived].mean().round(4))

In [ ]:
df_final.info()

---
## Feature Significance for Supply Chain Optimization

| Feature | Why It Matters |
|---------|----------------|
| **Delay** | Directly quantifies delivery performance. Positive values flag SLA breaches, enabling carrier benchmarking. |
| **Distance** (Haversine) | Physics-based proxy for transit time, fuel consumption, and cold-chain exposure. Longer routes increase probability of temperature excursions. |
| **RouteRisk** | Fuses delay severity, geographic exposure, and late-delivery risk into a single continuous score. Continuous scores are more informative for gradient-based ML models (Hastie et al., 2009). |
| **TempDev** | Simulates cold-chain instability. In pharma/food supply chains, each additional degree-hour of deviation accelerates spoilage non-linearly (Mercier et al., 2017). |
| **QualityDegradation** | Models first-order kinetic degradation $Q_0 \cdot e^{-\lambda t}$ (Labuza & Riboh, 1982). Output on 0-100 scale serves as regression target or classification threshold. |
| **RefrigerationCost** | Captures cost trade-off between speed and refrigeration intensity. Same-day shipments need active cooling with tighter tolerances, increasing per-km cost. Enables total-cost-of-ownership optimisation. |